# Part B: Reinforcement-Learning

## Project Overview

This notebook will guide you through implementing a single-agent reinforcement learning algorithm using stable-baselines3. 

In Part B, you will implement a **single-agent reinforcement learning** approach to exploration using **Stable-Baselines3** (PPO algorithm).

### Key Concepts

Unlike the heuristic approach in Part A, the RL agent learns exploration strategies through trial and error. The agent:
- Receives **egocentric observations** (local view around itself)
- Learns to maximize **coverage** while avoiding **collisions**
- Uses **reward shaping** to guide learning
- Trains with **curriculum learning** (easy → hard maps)

### What You'll Implement

You will complete functions in the `b_reinforcement_learning/` module:

1. **Curriculum Map Generation** (`curriculum.py`) - Level 0 (Vacuum), Level 1 (Sparse), Level 2 (Maze)
2. **Reward Components** (`rewards.py`) - Extrinsic, Safety, Intrinsic
3. **Gymnasium Environment** (`gym_env.py`) - Egocentric observations, history tracking
4. **Training Pipeline** (`train.py`) - PPO training with curriculum progression

This notebook will guide you through each component with testing, training, and evaluation.

## Notebook Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%env PYTORCH_ENABLE_MPS_FALLBACK=1

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
import gymnasium as gym
from stable_baselines3 import PPO

import torch

from b_reinforcement_learning.curriculum import CurriculumMapGenerator
from b_reinforcement_learning.rewards import RewardCalculator
from b_reinforcement_learning.gym_env import ExplorationEnv
from b_reinforcement_learning.train import (
    train_curriculum_level,
    train_from_scratch, 
    continue_training,
    plot_training_curves,
    evaluate_model,
    full_curriculum_training
)
from b_reinforcement_learning.custom_cnn import SmallCNN, create_policy_kwargs

# Import simulation for reference
from simulation.types import CellState

print("✓ All imports successful!")

## Step 1: Understanding Curriculum Learning

We'll train the agent progressively on three difficulty levels:

**Level 0: The Vacuum** - Empty environment with only border walls  
**Level 1: Sparse Obstacles** - Random obstacles (10% density)  

This curriculum allows the agent to learn basic behaviors first, then adapt to complex scenarios.

### 🎯 Your Task

Open `b_reinforcement_learning/curriculum.py` and implement the two map generators:

1. **`generate_level_0_vacuum()`** - Create an empty grid with only border walls
2. **`generate_level_1_sparse()`** - Randomly place obstacles with specified density

**Requirements:**
- All maps should have borders as OCCUPIED
- Robot should start in a FREE cell
- Maps use the same encoding as simulation (FREE=0, OCCUPIED=1, UNKNOWN=-1)

Once implemented, test your generators below.

In [ ]:
# Test the curriculum map generator
map_gen = CurriculumMapGenerator(size=20, seed=42)

# Visualize all three levels
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for level in range(2):
    truth, robot_start = map_gen.generate(level)
    
    ax = axes[level]
    
    # Create visualization
    img = np.zeros((20, 20, 3), dtype=np.uint8)
    img[truth == int(CellState.FREE)] = [255, 255, 255]  # White for free
    img[truth == int(CellState.OCCUPIED)] = [0, 0, 0]  # Black for obstacles
    img[robot_start[1], robot_start[0]] = [255, 0, 0]  # Red for robot
    
    ax.imshow(img, interpolation='nearest')
    ax.set_title(f'Level {level}', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Calculate metrics
    free_cells = np.sum(truth == int(CellState.FREE))
    occupied_cells = np.sum(truth == int(CellState.OCCUPIED))
    obstacle_density = occupied_cells / (free_cells + occupied_cells)
    
    print(f"Level {level}: Free={free_cells}, Occupied={occupied_cells}, "
          f"Density={obstacle_density:.2%}, Start={robot_start}")

plt.tight_layout()
plt.show()

print("\n✓ Curriculum map generation complete!")

## Step 2: Reward Function Components

The reward function guides the agent's learning. It consists of five components:

$$R_{total} = R_{extrinsic} + R_{safety} + R_{intrinsic} + R_{completion} - R_{step}$$

1. **Extrinsic Reward** ($R_{extrinsic} = \alpha \times \text{newly\_revealed}$): Reward for discovering new cells
2. **Safety Penalty** ($R_{safety} = -\beta$ if collision): Heavy penalty for hitting obstacles
3. **Intrinsic Reward** ($R_{intrinsic} = -\lambda \times \text{history}$): Penalty for revisiting areas (boredom)
4. **Completion Reward** $$R_{completion} = \begin{cases}
        \text{completion bonus} & \text{if reached threshold}\\
        0 & \text{otherwise}
    \end{cases}$$.
5. **Step Penalty**

### 🎯 Your Task

Open `b_reinforcement_learning/rewards.py` and implement the three reward functions (all should be very short):

1. **`calculate_extrinsic_reward()`** - Reward = α × newly_revealed
2. **`calculate_safety_penalty()`** - Penalty = -β if collision, else 0
3. **`calculate_intrinsic_reward()`** - Penalty = -λ × history_value

**Key Parameters:**
- α (alpha) = 1.0: Weight for information gain
- β (beta) = 15.0: Collision penalty (must be > max 1-step exploration reward)
- λ (lambda) = 0.5: Weight for history penalty

Once implemented, test below.

In [ ]:
# Test the reward calculator
reward_calc = RewardCalculator(alpha=1.0, beta=15.0, lambda_=0.5)

# Test scenario 1: Good exploration (revealed 3 cells, no collision, low history)
print("Scenario 1: Good exploration")
r_total, r_dict = reward_calc.calculate_total_reward(
    newly_revealed=3,
    collision=False,
    history_value=0.1
)
print(f"  Total reward: {r_total:.2f}")
print(f"  Components: {r_dict}")

# Test scenario 2: Collision
print("\nScenario 2: Collision with obstacle")
r_total, r_dict = reward_calc.calculate_total_reward(
    newly_revealed=0,
    collision=True,
    history_value=0.0
)
print(f"  Total reward: {r_total:.2f}")
print(f"  Components: {r_dict}")

# Test scenario 3: Revisiting area (high history)
print("\nScenario 3: Revisiting explored area")
r_total, r_dict = reward_calc.calculate_total_reward(
    newly_revealed=0,
    collision=False,
    history_value=0.9
)
print(f"  Total reward: {r_total:.2f}")
print(f"  Components: {r_dict}")

print("\n✓ Reward calculation tests complete!")

## Step 3: Gymnasium Environment

The `ExplorationEnv` class wraps our exploration task into a Gymnasium-compatible environment suitable for RL training.

### Key Features:

**Observation Space**: $(11, 11, 3)$ tensor (with radius=5) 
- Channel 0: Obstacles (1 = occupied, 0 = otherwise)
- Channel 1: Exploration status (1 = unknown, 0 = known)
- Channel 2: Trajectory history (0.0 to 1.0, recent visits higher)

**Action Space**: Discrete(4) - UP, DOWN, LEFT, RIGHT

**State Tracking**:
- `truth`: Ground truth map (hidden from agent)
- `fused_map`: Agent's knowledge (starts all UNKNOWN)
- `history_map`: Tracks visited locations with temporal decay

### 🎯 Your Task

Open `b_reinforcement_learning/gym_env.py` and implement key methods:

1. **`_sense_environment()`** - Reveal cells within sensor_radius (Manhattan distance)
2. **`_get_observation()`** - Construct 3-channel egocentric observation tensor

**Requirements:**
- Sensing uses L1/Manhattan distance: $|dx| + |dy| \leq radius$
- Observations are ego-centric (centered on robot)
- Out-of-bounds cells should be treated as obstacles in observations
- History map decays each step: `history = max(0, history - decay)`

Test your implementation below.

In [ ]:
# Test the Gymnasium environment
env = ExplorationEnv(size=30, sensor_radius=5, seed=42)

# Reset and get initial observation
obs, info = env.reset(options={'level': 0})

print(f"Observation shape: {obs.shape}")
print(f"Expected shape: (11, 11, 3) - channels-last for SB3")
print(f"Observation dtype: {obs.dtype}")
print(f"Observation range: [{obs.min():.2f}, {obs.max():.2f}]")
print(f"\nInitial info: {info}")

# Verify observation channels
print(f"\nChannel 0 (Obstacles): unique values = {np.unique(obs[:,:,0])}")
print(f"Channel 1 (Exploration): unique values = {np.unique(obs[:,:,1])}")
print(f"Channel 2 (History): min={obs[:,:,2].min():.2f}, max={obs[:,:,2].max():.2f}")

# Take a few random actions
print("\nTaking random actions:")
for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"  Step {i+1}: Action={action}, Reward={reward:.2f}, "
          f"Coverage={info['coverage']:.2%}, Collision={info['collision']}")

env.render('ascii')

print("\n✓ Environment test complete!")

### Visualize Observation Channels

Let's visualize what the agent actually "sees" at each step.

In [ ]:
# Visualize the observation channels
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

channel_names = ['Obstacles', 'Exploration (Unknown)', 'History']
cmaps = ['gray', 'Blues', 'hot']

for i in range(3):
    ax = axes[i]
    # Extract channel (channels-last format: height, width, channel)
    im = ax.imshow(obs[:, :, i], cmap=cmaps[i], interpolation='nearest')
    ax.set_title(f'Channel {i}: {channel_names[i]}', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

print("The agent receives these 3 channels (in channels-last format) as input to the CNN policy.")

### Validate Environment with SB3 Checker

Let's use Stable-Baselines3's environment checker to verify our implementation.

In [ ]:
# Run Stable-Baselines3 environment checker
from stable_baselines3.common.env_checker import check_env

print("Running SB3 environment checker...")
env_check = ExplorationEnv(size=20, sensor_radius=5, seed=42)

try:
    check_env(env_check, warn=True)
    print("✓ Environment passes SB3 checks!")
except Exception as e:
    print(f"⚠ Environment check failed: {e}")
    import traceback
    traceback.print_exc()

## Step 4: Training with PPO

Now we'll train the agent using Stable-Baselines3's PPO implementation with curriculum learning.

### Training Strategy:

1. **Level 0 (50k steps)**: Learn basic movement and exploration
2. **Level 1 (100k steps)**: Learn collision avoidance with sparse obstacles

Each level loads the previous level's model as initialization (transfer learning).

### Quick Training (Single Level Demo)

Let's first do a quick training run on Level 0 to see the training process.

In [ ]:
# Quick training demo on Level 0 (The Vacuum)
from b_reinforcement_learning.train import train_curriculum_level

# If you have a compatible GPU, you can set the device to "cuda" or "mps" for faster training. Otherwise, it will default to CPU.

print("Training a PPO agent on Level 0 for 50k steps...")
print("This should take 2-3 minutes.\n")

# Set device to 'cpu', or 'mps'/'cuda' if available
with torch.device("cpu"):
    model_demo, callback_demo = train_curriculum_level(
        level=0,
        timesteps=50_000,
        model_path=None,
        save_path="./models_demo",
        env_size=30,
        n_envs=5,  # Use 5 parallel environments
        verbose=1,
    )

print("\n✓ Demo training complete!")

In [ ]:
# Plot training curves for the demo
stats = callback_demo.get_statistics()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Rewards
if stats['rewards']:
    axes[0, 0].plot(stats['rewards'], alpha=0.6)
    if len(stats['rewards']) > 5:
        window = min(20, len(stats['rewards']) // 5)
        smoothed = np.convolve(stats['rewards'], np.ones(window)/window, mode='valid')
        axes[0, 0].plot(smoothed, linewidth=2, label='Smoothed')
    axes[0, 0].set_title('Episode Rewards')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# Coverage
if stats['coverages']:
    axes[0, 1].plot(stats['coverages'], alpha=0.6)
    if len(stats['coverages']) > 5:
        window = min(20, len(stats['coverages']) // 5)
        smoothed = np.convolve(stats['coverages'], np.ones(window)/window, mode='valid')
        axes[0, 1].plot(smoothed, linewidth=2, label='Smoothed')
    axes[0, 1].set_title('Episode Coverage')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Coverage %')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# Episode lengths
if stats['lengths']:
    axes[1, 0].plot(stats['lengths'], alpha=0.6)
    if len(stats['lengths']) > 5:
        window = min(20, len(stats['lengths']) // 5)
        smoothed = np.convolve(stats['lengths'], np.ones(window)/window, mode='valid')
        axes[1, 0].plot(smoothed, linewidth=2, label='Smoothed')
    axes[1, 0].set_title('Episode Lengths')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Steps')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Collisions
if stats['collisions']:
    axes[1, 1].plot(stats['collisions'], alpha=0.6)
    if len(stats['collisions']) > 5:
        window = min(20, len(stats['collisions']) // 5)
        smoothed = np.convolve(stats['collisions'], np.ones(window)/window, mode='valid')
        axes[1, 1].plot(smoothed, linewidth=2, label='Smoothed')
    axes[1, 1].set_title('Episode Collisions')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Collisions')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("You should see rewards and coverage increasing, while collisions decrease!")

## Step 5: Evaluate the Trained Agent

Let's test the trained agent and visualize its exploration behavior.

In [ ]:
# Evaluate the trained model
print("Evaluating trained agent on Level 0...")

with torch.device("mps"):
    eval_results = evaluate_model(
        model_path="./models_demo/ppo_level_0.zip",
        level=0,
        n_episodes=5,
        env_size=20,
        render=False,
    )

# Plot evaluation results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(len(eval_results['rewards'])), eval_results['rewards'], color='steelblue')
axes[0].set_title('Episode Rewards', fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(eval_results['coverages'])), eval_results['coverages'], color='forestgreen')
axes[1].set_title('Episode Coverage', fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Coverage %')
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% target')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].bar(range(len(eval_results['lengths'])), eval_results['lengths'], color='coral')
axes[2].set_title('Episode Lengths', fontweight='bold')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Steps')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualize Agent Exploration

Let's watch the trained agent explore step by step.

In [ ]:
# Visualize agent exploration


# Load the trained model
model = PPO.load("./models_demo/ppo_level_0.zip")

# Create environment
env = ExplorationEnv(size=20, sensor_radius=5, seed=123)
obs, info = env.reset(options={'level': 0})

# Run one episode and collect snapshots
snapshots = []
maps = env.get_maps()
snapshots.append((maps['fused_map'].copy(), env.robot_pos))

for step in range(100):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    
    if step % 10 == 0:  # Capture every 10 steps
        maps = env.get_maps()
        snapshots.append((maps['fused_map'].copy(), env.robot_pos))
    
    if terminated or truncated:
        # Capture final state
        maps = env.get_maps()
        snapshots.append((maps['fused_map'].copy(), env.robot_pos))
        break

print(f"Episode completed in {step+1} steps with {info['coverage']:.2%} coverage")

# Visualize snapshots
n_snapshots = len(snapshots)
cols = min(5, n_snapshots)
rows = (n_snapshots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
if rows == 1:
    axes = axes.reshape(1, -1)

for idx, (fused_map, robot_pos) in enumerate(snapshots):
    row = idx // cols
    col = idx % cols
    ax = axes[row, col] if rows > 1 else axes[col]
    
    # Create RGB visualization
    img = np.zeros((20, 20, 3), dtype=np.uint8)
    img[fused_map == int(CellState.UNKNOWN)] = [128, 128, 128]  # Gray
    img[fused_map == int(CellState.FREE)] = [255, 255, 255]  # White
    img[fused_map == int(CellState.OCCUPIED)] = [0, 0, 0]  # Black
    img[robot_pos[1], robot_pos[0]] = [255, 0, 0]  # Red for robot
    
    ax.imshow(img, interpolation='nearest')
    ax.set_title(f'Step {idx*10}' if idx < len(snapshots)-1 else 'Final', fontsize=10)
    ax.axis('off')

# Hide empty subplots
for idx in range(n_snapshots, rows * cols):
    row = idx // cols
    col = idx % cols
    ax = axes[row, col] if rows > 1 else axes[col]
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✓ You can see the agent progressively exploring the environment!")

### Lets Animate It!

In [ ]:
# Animated visualization of RL agent exploring
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Load trained model
model_anim = PPO.load("./models_demo/ppo_level_0.zip")

# Create environment
env_anim = ExplorationEnv(size=30, sensor_radius=5, seed=456)
obs_anim, info_anim = env_anim.reset(options={'level': 0})

# Run episode and collect all frames
print("Collecting frames for animation...")
frames = []
maps = env_anim.get_maps()
frames.append({
    'fused_map': maps['fused_map'].copy(),
    'robot_pos': tuple(env_anim.robot_pos),
    'coverage': info_anim['coverage'],
    'step': 0
})

max_steps = 200
for step in range(max_steps):
    action, _ = model_anim.predict(obs_anim, deterministic=True)
    obs_anim, reward, terminated, truncated, info_anim = env_anim.step(action)
    
    # Capture every frame
    maps = env_anim.get_maps()
    frames.append({
        'fused_map': maps['fused_map'].copy(),
        'robot_pos': tuple(env_anim.robot_pos),
        'coverage': info_anim['coverage'],
        'step': step + 1
    })
    
    if terminated or truncated:
        print(f"Episode completed in {step+1} steps with {info_anim['coverage']:.2%} coverage")
        break

print(f"Animation will show {len(frames)} frames")

# Create animation
fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

# Initialize with first frame
first_frame = frames[0]
fused_map = first_frame['fused_map']
robot_pos = first_frame['robot_pos']

img_data = np.zeros((30, 30, 3), dtype=np.uint8)
img_data[fused_map == int(CellState.UNKNOWN)] = [128, 128, 128]
img_data[fused_map == int(CellState.FREE)] = [255, 255, 255]
img_data[fused_map == int(CellState.OCCUPIED)] = [0, 0, 0]
img_data[robot_pos[1], robot_pos[0]] = [255, 0, 0]

im = ax.imshow(img_data, interpolation='nearest')
title = ax.set_title(f'Step {first_frame["step"]} | Coverage: {first_frame["coverage"]:.1%}', 
                      fontsize=14, fontweight='bold')

def update_frame(frame_idx):
    frame = frames[frame_idx]
    fused_map = frame['fused_map']
    robot_pos = frame['robot_pos']
    
    # Create RGB visualization
    img_data = np.zeros((30, 30, 3), dtype=np.uint8)
    img_data[fused_map == int(CellState.UNKNOWN)] = [128, 128, 128]  # Gray
    img_data[fused_map == int(CellState.FREE)] = [255, 255, 255]     # White
    img_data[fused_map == int(CellState.OCCUPIED)] = [0, 0, 0]       # Black
    img_data[robot_pos[1], robot_pos[0]] = [255, 0, 0]               # Red robot
    
    # Update the image data
    im.set_data(img_data)
    title.set_text(f'Step {frame["step"]} | Coverage: {frame["coverage"]:.1%}')
    
    return im, title

# Create animation with skip frames for smoother playback
skip = max(1, len(frames) // 100)  # Show ~100 frames max
frame_indices = list(range(0, len(frames), 1))

anim = FuncAnimation(fig, update_frame, frames=frame_indices, 
                     interval=50, blit=True, repeat=True)

# Display animation
print("\n🎬 Playing animation...")
print(f"Showing {len(frame_indices)} frames (skipping every {skip})")
print("(The animation will loop continuously)")
HTML(anim.to_jshtml())

## Step 6: Full Curriculum Training (Optional)

For complete training, run the full curriculum across all three levels. This will take significant time (30-60 minutes depending on your hardware).

**Warning**: This is computationally intensive. Only run if you have time and want the best results.

In [ ]:
# OPTIONAL: Full curriculum training
# Uncomment to run (this will take 30-60 minutes!)

# Use "cuda" if you have an NVIDIA GPU, or "mps" for Apple Silicon
# Otherwise will take a long time

with torch.device("cpu"):
    final_model, all_callbacks = full_curriculum_training(
        level_0_steps=50_000,
        level_1_steps=200_000,
        env_size=30,
        save_dir="./models_full",
    )

# print("Skipping full curriculum training in this demo.")
# print("To run it, uncomment the code above.")

## Step 6.b: Evaluate Model

In [ ]:
# Animated visualization of RL agent exploring
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Load trained model
model_anim = PPO.load("./models_full/ppo_level_1.zip")

# Create environment
env_anim = ExplorationEnv(size=50, sensor_radius=5, seed=5223213126)
obs_anim, info_anim = env_anim.reset(options={'level': 1})

# Run episode and collect all frames
print("Collecting frames for animation...")
frames = []
maps = env_anim.get_maps()
frames.append({
    'fused_map': maps['fused_map'].copy(),
    'robot_pos': tuple(env_anim.robot_pos),
    'coverage': info_anim['coverage'],
    'step': 0
})

max_steps = 1000
for step in range(max_steps):
    action, _ = model_anim.predict(obs_anim, deterministic=False)
    obs_anim, reward, terminated, truncated, info_anim = env_anim.step(action)
    
    # Capture every frame
    maps = env_anim.get_maps()
    frames.append({
        'fused_map': maps['fused_map'].copy(),
        'robot_pos': tuple(env_anim.robot_pos),
        'coverage': info_anim['coverage'],
        'step': step + 1
    })
    
    if terminated or truncated:
        print(f"Episode completed in {step+1} steps with {info_anim['coverage']:.2%} coverage")
        break

print(f"Animation will show {len(frames)} frames")

# Create animation
fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

# Initialize with first frame
first_frame = frames[0]
fused_map = first_frame['fused_map']
robot_pos = first_frame['robot_pos']

img_data = np.zeros((50, 50, 3), dtype=np.uint8)
img_data[fused_map == int(CellState.UNKNOWN)] = [128, 128, 128]
img_data[fused_map == int(CellState.FREE)] = [255, 255, 255]
img_data[fused_map == int(CellState.OCCUPIED)] = [0, 0, 0]
img_data[robot_pos[1], robot_pos[0]] = [255, 0, 0]

im = ax.imshow(img_data, interpolation='nearest')
title = ax.set_title(f'Step {first_frame["step"]} | Coverage: {first_frame["coverage"]:.1%}', 
                      fontsize=14, fontweight='bold')

def update_frame(frame_idx):
    frame = frames[frame_idx]
    fused_map = frame['fused_map']
    robot_pos = frame['robot_pos']
    
    # Create RGB visualization
    img_data = np.zeros((50, 50, 3), dtype=np.uint8)
    img_data[fused_map == int(CellState.UNKNOWN)] = [128, 128, 128]  # Gray
    img_data[fused_map == int(CellState.FREE)] = [255, 255, 255]     # White
    img_data[fused_map == int(CellState.OCCUPIED)] = [0, 0, 0]       # Black
    img_data[robot_pos[1], robot_pos[0]] = [255, 0, 0]               # Red robot
    
    # Update the image data
    im.set_data(img_data)
    title.set_text(f'Step {frame["step"]} | Coverage: {frame["coverage"]:.1%}')
    
    return im, title

# Create animation with skip frames for smoother playback
skip = max(1, len(frames) // 100)  # Show ~100 frames max
frame_indices = list(range(0, len(frames), 1))

anim = FuncAnimation(fig, update_frame, frames=frame_indices, 
                     interval=50, blit=True, repeat=True)

# Display animation
print("\n🎬 Playing animation...")
print(f"Showing {len(frame_indices)} frames (skipping every {skip})")
print("(The animation will loop continuously)")
HTML(anim.to_jshtml())

If the model did not perform well enough, you can continue to train it using the cell below.

In [ ]:
# Example 2: Continue training an existing model
# Train the level 1 model for an additional 200k steps
model_level1_continued, callback_continued = continue_training(
    model_path="./models_full/ppo_level_1.zip",
    level=1,
    additional_timesteps=100_000,
    # save_path=None means it will overwrite the original model
    # Or specify a different path to keep both versions
    save_path="./models_custom_cont",
    env_size=20,
    n_envs=4,
    learning_rate=1e-5
)

print("\n✓ Continued training complete!")

## Step 7: Analysis and Extensions

Let's analyze the trained agent's behavior and discuss potential improvements.

### Key Observations

After training, you should observe:

1. **Level 0 (Vacuum)**: Agent learns to explore systematically (often wall-following or spiral patterns)
2. **Level 1 (Sparse)**: Agent learns to avoid obstacles while maintaining coverage

### Reward Shaping Insights

The three reward components work together:

- **Extrinsic** drives exploration
- **Safety** prevents crashes
- **Intrinsic** (history) prevents loops and encourages coverage

Without intrinsic reward, agents often get stuck in local loops!

### Potential Extensions

1. **Multi-Agent RL**: Extend to cooperative multi-agent exploration
2. **Recurrent Policies**: Use LSTM to maintain memory of visited areas
3. **Attention Mechanisms**: Learn to focus on important features
4. **Curriculum Variations**: Try different map generators
5. **Transfer Learning**: Test on unseen map types
6. **Hierarchical RL**: High-level goal selection + low-level navigation